# EEG Classification with SVM

## Imports

In [27]:
import os
import mne
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

## Load Participant Labels

In [29]:
participants = pd.read_csv("openneuro_data/participants.tsv", sep="\t")

## Define Band Power Function

In [31]:
def band_power(raw, low, high):
    band_raw = raw.copy().filter(low, high, verbose=False)
    band_data = band_raw.get_data()
    return np.mean(band_data ** 2)

## Extract EEG Features

In [ ]:
rows = []

for _, row in participants.iterrows():
    subject = row["participant_id"]
    label = row["Group"]

    file_path = f"openneuro_data/{subject}/eeg/{subject}_task-eyesclosed_eeg.set"

    if not os.path.exists(file_path):
        print(f"Missing file for {subject}")
        continue

    try:
        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

        delta = band_power(raw, 0.5, 4)
        theta = band_power(raw, 4, 8)
        alpha = band_power(raw, 8, 12)
        beta = band_power(raw, 12, 30)

        rows.append({
            "participant_id": subject,
            "delta": delta,
            "theta": theta,
            "alpha": alpha,
            "beta": beta,
            "label": label
        })

        print(f"Done: {subject}")

    except Exception as e:
        print(f"Error with {subject}: {e}")

Done: sub-001
Done: sub-002
Done: sub-003


/var/folders/17/kswrw_hs6hj1r3bn075438d40000gn/T/ipykernel_21950/2059419214.py:14: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/var/folders/17/kswrw_hs6hj1r3bn075438d40000gn/T/ipykernel_21950/2059419214.py:14: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Done: sub-004
Done: sub-005


/var/folders/17/kswrw_hs6hj1r3bn075438d40000gn/T/ipykernel_21950/2059419214.py:14: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Done: sub-006


...

/var/folders/17/kswrw_hs6hj1r3bn075438d40000gn/T/ipykernel_21950/2059419214.py:14: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Done: sub-010


/var/folders/17/kswrw_hs6hj1r3bn075438d40000gn/T/ipykernel_21950/2059419214.py:14: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Done: sub-011


/var/folders/17/kswrw_hs6hj1r3bn075438d40000gn/T/ipykernel_21950/2059419214.py:14: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


## Preview Data

In [ ]:
features_df = pd.DataFrame(rows)

print(features_df.shape)
print(features_df.head())
print(features_df["label"].value_counts())

## Prepare Data

In [ ]:
X = features_df[["delta", "theta", "alpha", "beta"]]
y = features_df["label"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label mapping:")
for original, encoded in zip(le.classes_, range(len(le.classes_))):
    print(f"{original} -> {encoded}")

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

## Build and Train SVM

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluate Results
print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## Alpha vs Theta Scatter Plot for Alzheimer's and Control Groups

In [ ]:
name_map = {
    "A": "Alzheimer's",
    "C": "Control"
}

ac_df = features_df[features_df["label"].isin(["A", "C"])].copy()

plt.figure(figsize=(8, 6))

for label in ac_df["label"].unique():
    subset = ac_df[ac_df["label"] == label]
    plt.scatter(
        subset["alpha"],
        subset["theta"],
        label=name_map[label],
        alpha=0.7
    )

plt.xlabel("Alpha Power")
plt.ylabel("Theta Power")
plt.title("Alpha vs Theta by Group")
plt.legend()
plt.grid(True)
plt.show()